# Colab 3
## K-means and PCA

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyspark
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark import SparkContext, SparkConf
from pyspark.ml.linalg import Vectors
from sklearn.datasets import load_breast_cancer
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import PCA
from pyspark.ml.linalg import Vectors

In [2]:
breast_cancer = load_breast_cancer()

In [3]:
conf = SparkConf().set("spark.ui.port", "4050")

In [4]:
# create the context
sc = pyspark.SparkContext(conf=conf)
spark = SparkSession.builder.getOrCreate()
print(spark.sparkContext.master)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/03 23:17:28 WARN Utils: Your hostname, panteliss-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.255.221.4 instead (on interface en0)
26/02/03 23:17:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/03 23:17:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/03 23:17:30 WARN Utils: Service 'SparkUI' could not bind on port 4050. Attempting port 4051.
26/02/03 23:17:30 WARN Utils: Service 'SparkUI' could not bind on port 4051. Attempting port 4052.


local[*]


### Load breast cancer data as dataframe

In [5]:
pd_df = pd.DataFrame(breast_cancer.data, columns=breast_cancer.feature_names)
df = spark.createDataFrame(pd_df)

In [6]:
def set_df_columns_nullable(spark, df, column_list, nullable=False):
    for struct_field in df.schema:
        if struct_field.name in column_list:
            struct_field.nullable = nullable
    df_mod = spark.createDataFrame(df.rdd, df.schema)
    return df_mod

In [7]:
df = set_df_columns_nullable(spark, df, df.columns)
df = df.withColumn('features', array(df.columns))
vectors = df.rdd.map(lambda row: Vectors.dense(row.features))

In [8]:
features = spark.createDataFrame(vectors.map(Row), ["features"])
labels = pd.Series(breast_cancer.target)

In [9]:
df.printSchema()

root
 |-- mean radius: double (nullable = false)
 |-- mean texture: double (nullable = false)
 |-- mean perimeter: double (nullable = false)
 |-- mean area: double (nullable = false)
 |-- mean smoothness: double (nullable = false)
 |-- mean compactness: double (nullable = false)
 |-- mean concavity: double (nullable = false)
 |-- mean concave points: double (nullable = false)
 |-- mean symmetry: double (nullable = false)
 |-- mean fractal dimension: double (nullable = false)
 |-- radius error: double (nullable = false)
 |-- texture error: double (nullable = false)
 |-- perimeter error: double (nullable = false)
 |-- area error: double (nullable = false)
 |-- smoothness error: double (nullable = false)
 |-- compactness error: double (nullable = false)
 |-- concavity error: double (nullable = false)
 |-- concave points error: double (nullable = false)
 |-- symmetry error: double (nullable = false)
 |-- fractal dimension error: double (nullable = false)
 |-- worst radius: double (nullable

### K-Means
With k=2, train the k-means algortithm in spark

In [10]:
kmeans = KMeans().setK(2).setSeed(1)
model = kmeans.fit(features)

26/02/03 23:19:10 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


In [11]:
predictions = model.transform(features)

In [12]:
evaluator = ClusteringEvaluator()

In [13]:
silhouette = evaluator.evaluate(predictions)
print("Silhouette with squared euclidean distance = " + str(silhouette))

Silhouette with squared euclidean distance = 0.8342904262826145


### Collect all predictions to the driver

In [ ]:
rows = predictions.select('prediction').collect()
preds = [row.prediction for row in rows]

### Compare predictions vs actual labels

In [15]:
err = np.count_nonzero(np.array(preds) != np.array(labels))
print(f'Misclustered examples: {err} out of {len(labels)}')

Misclustered examples: 83 out of 569


### Reduce data dimensionality with PCA

In [16]:
pca = PCA(k = 2, inputCol="features", outputCol="pcaFeatures")
pca_model = pca.fit(features)
features_pca = pca_model.transform(features).select("pcaFeatures").withColumnRenamed("pcaFeatures", "features")

26/02/03 23:20:43 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


In [17]:
# DEBUG: Verify dimensions
print("Original feature size:", len(features.first()['features']))
print("PCA feature size:", len(features_pca.first()['features']))

Original feature size: 30
PCA feature size: 2


### Run k-means on the reduced, lower dimensionality points

In [18]:
kmeans = KMeans().setK(2).setSeed(1)
model = kmeans.fit(features_pca)

In [19]:
predictions_pca = model.transform(features_pca)

In [20]:
evaluator = ClusteringEvaluator()
silhouette = evaluator.evaluate(predictions_pca)
print("Silhouette with squared euclidean distance (PCA) = " + str(silhouette))

Silhouette with squared euclidean distance (PCA) = 0.8348610363444836


In [21]:
rows = predictions_pca.select('prediction').collect()
preds_pca = [row.prediction for row in rows]

In [22]:
# Compare predictions vs labels (mismatches)
err = np.count_nonzero(np.array(preds_pca) != np.array(labels))
print(f'Misclustered examples (PCA): {err} out of {len(labels)}')

Misclustered examples (PCA): 83 out of 569
